# Transformações Power BI — Siplan RPS

Substitui as transformações do **Power Query** por Python puro (pandas/numpy), mantendo a lógica de negócio do RPS Sesc SP.

## Arquitetura

```
HIVE (siplan_data, siplan_solicitacao, siplan_acao, projetos, ...)
    ↓  Dataflow Gen2  ← os SQLs deste notebook viram queries aqui
LH_Siplan_Staging     ← tabelas raw (uma por fonte)
    ↓  Fabric Notebook ← este notebook, lendo com spark.table()
LH_Siplan_Prod        ← tabelas finais consumidas pelo Power BI
```

> **Desenvolvimento local:** as queries rodam via `pyodbc` (Kerberos/Hive). No Fabric, `query_to_df(sql)` é substituído por `spark.table('raw_xxx').toPandas()`.

## Tabelas staging e DataFrames (ordem de execução)

| # | DataFrame | Fonte Hive | Observações |
|---|---|---|---|
| 1a | `raw_acoes_df` | `siplan_acao` + `projetos` | Campos de faixa etária incluídos: `idade_inicial`, `idade_final`, `recomendacao_etaria`, `linguagem`, `estimativa_publico`, `lugares` |
| 1b | `raw_tags_df` | `siplan_tag` + `siplan_projeto_tag` | Inclui `todas_as_tags` (pipe-separated, calculado em Python) |
| 1c | `rps_parcial_df` | `raw_acoes_df` + `raw_tags_df` + passagens | Campos: `atividade_id`, `servico`, `subatividade`, `tag`, `tem_passagem` |
| 2 | `datas_df` | `siplan_data` (agregado por atividade) | `primeiradata`, `ultimadata`, `totalSessoes`, `totalHoras`, `autonomiaTemporal` |
| 3 | `contracts_df` | `siplan_solicitacao` + `siplan_acao` | Contratos administrativos; origem: `autonomiaCusto`. **Será revisado para usar `raw_solicitacoes_df` como fonte** |
| 4 | `autonomias_df` | `datas_df` + `contracts_df` + `rps_parcial_df` | Hierarquia DIREG (3) > STS (2) > UO (1) |
| 5 | `raw_projetos_df` | `siplan_acao` + `projetos` + `projeto_uo` | Deduplicado por `atividade_id`; campo `tem_pai` (bool) |
| 6 | `raw_datas_sessoes_df` | `siplan_data` (detalhe por sessão) | Uma linha por sessão; campos de data, hora, `TipoLocal`, `PeriodoDia` |
| 7 | `raw_acessibilidade_df` | `siplan_acessibilidade` | Uma linha por recurso; `tem_dispositivo` sempre "sim" (id=9 excluído na query) |
| 8 | `raw_solicitacoes_df` | `siplan_solicitacao` | Staging pura: todos os custos > 0 do período; fonte para as saídas 8a e futuras |
| 8a | `solicitacoes_desc_df` | `raw_solicitacoes_df` | Uma linha por `atividade_id`; campo `item_desc` com custos concatenados (normalizados) |

## Campos derivados pendentes (tabela base — seção futura)

| Campo | Fonte | Lógica |
|---|---|---|
| `faixa` | `raw_acoes_df` | `np.select` com `idade_inicial/final`, `linguagem`, `recomendacao_etaria` |
| `tem_dispositivo` | `raw_acessibilidade_df` | "sim" se `atividade_id` presente, "não" caso contrário |
| `espaco_brincar` | `raw_acoes_df` + `raw_tags_df` + `raw_projetos_df` + `raw_datas_sessoes_df` | OR de ocorrências do nome em nome/tag/projeto/local |

## Normalização de `item_de_custo` (seção 8)

Categorias finais após normalização:
`Contrato PJ` · `Contrato PF` · `Contrato Cooperativa` · `Hospedagem` · `Passagem Aérea` · `Turismo` · `Transporte` · `Exibição de Filmes` · `Ação esportiva e recreativa` · `Contratações diversas` · `Compras` · `Alimentação` · `Sonorização` · `Iluminação` · `Audiovisual` · `Locação` · `Comunicação` · `Acessibilidade` · `Outros`

Descartados: `Camarim` (tipos 1/2), `Água`, `Verificar`, `Estagiário`


In [3]:
import warnings
import numpy as np
import pandas as pd
import pyodbc
from pathlib import Path

OUTPUT_PATH = Path('output')
OUTPUT_PATH.mkdir(exist_ok=True)
DATA_INICIAL = '2026-01-01'

# Ajuste a string abaixo se o DSN exigir parâmetros adicionais.
# Para autenticação Kerberos, normalmente basta usar o DSN configurado.
KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Authentication=Kerberos;'
# Se o driver exigir Trusted Connection no Windows, use:
# KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Trusted_Connection=Yes;'


def get_connection() -> pyodbc.Connection:
    return pyodbc.connect(KERBEROS_CONN_STR, autocommit=True)


def query_to_df(query: str) -> pd.DataFrame:
    with get_connection() as conn:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                message='.*SQLAlchemy connectable.*',
                category=UserWarning,
                module='pandas.io.sql',
            )
            return pd.read_sql_query(query, conn)


## 1. Ações

### 1a. Ações (siplan_acao + projetos) → `raw_acoes_df`

Base de ações programáticas com dados cadastrais, filtradas pelo período corrente.
Substitui `vw_siplan_acao_2025_26` — sem datas hardcoded, filtro via `DATA_INICIAL`.

**Campos:**
- `uo`, `status_atividade`, `atividade_id`, `nome`, `complemento` — identificação
- `uso_interno`, `manutencao`, `regular`, `integracao_sgc` — flags de gestão
- `recomendacao_etaria`, `produtor`, `tem_parceria`, `contatofornecedores`
- `areaprog`, `atividade`, `subatividade`, `servico` — hierarquia programática
- `tipo`, `subtipo`, `formato`, `linguagem` — classificação
- `gratuito` — `'Sim'`/`'Não'` calculado a partir de `precificacao`/`precificacao_desc`
- `precificacao_desc`, `institucional`, `projeto` — dados complementares

> Lakehouse Staging: **`raw_acoes`**

### 1b. Tags (siplan_tag + siplan_projeto_tag) → `raw_tags_df`

Tags por `atividade_id`, unindo tags diretas da ação e tags herdadas do projeto (UNION).
Substitui `vw_siplan_tags`.

**Campos:**
- `atividade_id`, `tag_nome`, `tag_grupo`, `tag_origem` — uma linha por atividade × tag
- `todas_as_tags` — todas as tags da atividade concatenadas com `" | "` (calculado em Python)

> Lakehouse Staging: **`raw_tags`**

### 1c. RPS Parcial (+ passagens) → `rps_parcial_df`

Derivado de `raw_acoes_df` + `raw_tags_df` + passagens aéreas.

- `tag` — `'Avaliação STS'` se a atividade tem essa tag (filtrado de `raw_tags_df`)
- `tem_passagem` — `'Sim'` se há solicitação de passagem aérea (via `siplan_solicitacao`)

Usado nas etapas seguintes para:
- Ajuste da `autonomiaTemporal` em **Datas** (servico/subatividade)
- Cálculo de `autonomiaCusto` em **Contratos** (servico/subatividade)
- Cálculo da `autonomia` final em **Autonomias** (tag + tem_passagem)

> Lakehouse Staging: **`raw_rps_parcial`**

In [ ]:
# Lakehouse Staging: raw_acoes
sql_raw_acoes = f'''
SELECT
    a.uo,
    a.status_atividade,
    a.atividade_id,
    a.nome,
    a.complemento,
    a.uso_interno,
    a.manutencao,
    a.regular,
    a.integracao_sgc,
    a.recomendacao_etaria,
    a.produtor,
    a.tem_parceria,
    a.contatofornecedores,
    a.areaprogramatica_nome                                            AS areaprog,
    a.desc_subprograma                                                 AS atividade,
    a.desc_modalidade                                                  AS subatividade,
    a.desc_realizacao                                                  AS servico,
    a.tipo_descricao                                                   AS tipo,
    a.subtipo_descricao                                                AS subtipo,
    a.formato,
    a.linguagem,
    a.idade_inicial,
    a.idade_final,
    a.estimativa_publico,
    a.lugares,
    CASE
        WHEN a.precificacao = 'Gratuito'
          OR LOWER(a.precificacao_desc) LIKE '%grátis%'
          OR REGEXP_EXTRACT(a.precificacao_desc, 'R\\$([0-9]+\\.?[0-9]*)') = '0'
        THEN 'Sim'
        ELSE 'Não'
    END                                                                AS gratuito,
    a.precificacao_desc,
    p.institucional,
    p.projeto
FROM stg_estatistico.siplan_acao a
LEFT JOIN stg_estatistico.projetos p ON a.projeto_id = p.projeto_id
WHERE a.atividade_id IS NOT NULL
  AND a.atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''

# Lakehouse Staging: raw_tags  (substitui vw_siplan_tags)
# todas_as_tags é calculada em Python após o load
sql_raw_tags = '''
SELECT
    acao.atividade_id                                                  AS atividade_id,
    siplan_tags.tag_nome                                               AS tag_nome,
    siplan_tags.tag_grupo                                              AS tag_grupo,
    siplan_tags.tag_origem                                             AS tag_origem
FROM estatistico.siplan_acao acao
LEFT JOIN (
    SELECT tag.atividade_id, tag.tag_nome, tag.grupo
    FROM stg_estatistico.siplan_tag tag
) acao_tag ON acao.atividade_id = acao_tag.atividade_id
LEFT JOIN (
    SELECT DISTINCT
        siplan_tag.atividade_id,
        siplan_tag.tag_nome,
        CASE WHEN siplan_tag.tag_grupo = 'Mobilização'
             THEN siplan_tag.tag_nome
             ELSE siplan_tag.tag_grupo
        END                                                            AS tag_grupo,
        CASE WHEN siplan_tag_na_acao.atividade_id IS NULL
             THEN 'acao'
             ELSE 'projeto'
        END                                                            AS tag_origem
    FROM (
        SELECT tag.atividade_id, tag.tag_nome, tag.grupo AS tag_grupo
        FROM stg_estatistico.siplan_tag tag
        UNION
        SELECT acao_proj_tag.atividade_id, proj_tag.tag_nome, proj_tag.tag_grupo
        FROM stg_estatistico.siplan_acao acao_proj_tag
        INNER JOIN (
            SELECT siplan_proj_tag.projeto_id, siplan_proj_tag.tag_nome,
                   siplan_proj_tag.grupo AS tag_grupo
            FROM stg_estatistico.siplan_projeto_tag siplan_proj_tag
            WHERE siplan_proj_tag.projeto_id IN (
                SELECT proj_tag.projeto_id
                FROM stg_estatistico.siplan_projeto_tag proj_tag
                GROUP BY proj_tag.projeto_id
                HAVING COUNT(*) <= 5
            )
        ) proj_tag ON acao_proj_tag.projeto_id = proj_tag.projeto_id
    ) siplan_tag
    LEFT JOIN (
        SELECT tag.atividade_id, tag.tag_nome, tag.grupo AS tag_grupo
        FROM stg_estatistico.siplan_tag tag
    ) siplan_tag_na_acao
        ON siplan_tag.atividade_id = siplan_tag_na_acao.atividade_id
       AND siplan_tag.tag_nome     = siplan_tag_na_acao.tag_nome
) siplan_tags ON acao.atividade_id = siplan_tags.atividade_id
WHERE siplan_tags.tag_nome IS NOT NULL
'''

# Passagens aéreas (para rps_parcial_df)
sql_passagens = '''
SELECT DISTINCT atividade_id
FROM stg_estatistico.siplan_solicitacao
WHERE item_grupo LIKE '%Passagem%'
'''

In [11]:
# --- 1a. raw_acoes_df ---
raw_acoes_df = query_to_df(sql_raw_acoes)
raw_acoes_df.columns    = [col.split('.')[-1] for col in raw_acoes_df.columns]
raw_acoes_df['atividade_id'] = raw_acoes_df['atividade_id'].astype(str).str.strip()
raw_acoes_df['servico']      = raw_acoes_df['servico'].astype(str).str.strip()
raw_acoes_df['subatividade'] = raw_acoes_df['subatividade'].astype(str).str.strip()

# --- 1b. raw_tags_df (+ todas_as_tags calculada em Python) ---
raw_tags_df = query_to_df(sql_raw_tags)
raw_tags_df.columns = [col.split('.')[-1] for col in raw_tags_df.columns]
raw_tags_df['atividade_id'] = raw_tags_df['atividade_id'].astype(str).str.strip()

todas_tags = (
    raw_tags_df.dropna(subset=['tag_nome'])
    .groupby('atividade_id')['tag_nome']
    .apply(lambda x: ' | '.join(sorted(x.unique())))
    .reset_index(name='todas_as_tags')
)
raw_tags_df = raw_tags_df.merge(todas_tags, on='atividade_id', how='left')

# --- 1c. rps_parcial_df (raw_acoes_df + raw_tags_df + passagens) ---
sts_df = (
    raw_tags_df[raw_tags_df['tag_nome'] == 'Avaliação STS'][['atividade_id']]
    .assign(tag='Avaliação STS')
    .drop_duplicates()
)

passagens_df = query_to_df(sql_passagens)
passagens_df.columns = [col.split('.')[-1] for col in passagens_df.columns]
passagens_df['atividade_id'] = passagens_df['atividade_id'].astype(str).str.strip()
passagens_df['tem_passagem'] = 'Sim'

rps_parcial_df = (
    raw_acoes_df[['atividade_id', 'servico', 'subatividade']]
    .merge(sts_df[['atividade_id', 'tag']], on='atividade_id', how='left')
    .merge(passagens_df[['atividade_id', 'tem_passagem']], on='atividade_id', how='left')
    .assign(
        tag          = lambda d: d['tag'].fillna(''),
        tem_passagem = lambda d: d['tem_passagem'].fillna('Não'),
    )
    .drop_duplicates(subset=['atividade_id'])
)

print(f'raw_acoes_df:   {raw_acoes_df.shape}')
print(f'raw_tags_df:    {raw_tags_df.shape}')
print(f'rps_parcial_df: {rps_parcial_df.shape}')

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_20632\2510096508.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)
C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_20632\2510096508.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)
C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_20632\2510096508.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


raw_acoes_df:   (83031, 25)
raw_tags_df:    (804716, 5)
rps_parcial_df: (33076, 5)


## 2. Datas (siplan_data)

Tabela de referência de datas por `atividade_id`, construída diretamente de `siplan_data` — **sem depender da view** `vw_siplan_datas_totais`.

Consultada **uma única vez** aqui. Toda a lógica da view foi trazida para este SQL.

**Campos produzidos:**
- `primeiradata`, `ultimadata` — primeira e última sessão da atividade
- `qt_sessoes` — número de sessões únicas (`sessao_id` distintos)
- `qt_datas_distintas` — número de datas distintas com sessão (dias de calendário em que há pelo menos uma sessão)
- `qt_horas` — total de horas (soma de `datafinal − datainicio` de cada sessão)
- `tempo_sessao` — duração média por sessão (`qt_horas / qt_sessoes`)
- `diascorridos` — dias corridos entre primeira e última sessão (`DATEDIFF`)
- `` `30dias` `` — 1 se `diascorridos > 30`
- `` `90dias` `` — 1 se `diascorridos > 90`
- `` `60horas` `` — 1 se `qt_horas > 60`
- `ano_2018` … `ano_2026` — flag `1/0` por ano (2020 e 2021 incluídos; exclusão feita nos relatórios)

> Lakehouse Staging: **`raw_datas`**

In [12]:
# Lakehouse Staging: raw_datas
sql_datas = '''
SELECT
    atividade_id,
    MIN(datainicio)                                                                AS primeiradata,
    MAX(datainicio)                                                                AS ultimadata,
    COUNT(DISTINCT sessao_id)                                                      AS qt_sessoes,
    COUNT(DISTINCT TO_DATE(datainicio))                                            AS qt_datas_distintas,
    SUM((UNIX_TIMESTAMP(datafinal) - UNIX_TIMESTAMP(datainicio)) / 3600)           AS qt_horas,
    SUM((UNIX_TIMESTAMP(datafinal) - UNIX_TIMESTAMP(datainicio)) / 3600)
        / COUNT(DISTINCT sessao_id)                                                AS tempo_sessao,
    DATEDIFF(MAX(datainicio), MIN(datainicio))                                      AS diascorridos,
    CASE WHEN DATEDIFF(MAX(datainicio), MIN(datainicio)) > 30  THEN 1 ELSE 0 END   AS `30dias`,
    CASE WHEN DATEDIFF(MAX(datainicio), MIN(datainicio)) > 90  THEN 1 ELSE 0 END   AS `90dias`,
    CASE WHEN SUM((UNIX_TIMESTAMP(datafinal) - UNIX_TIMESTAMP(datainicio)) / 3600)
              > 60 THEN 1 ELSE 0 END                                               AS `60horas`,
    MAX(CASE WHEN YEAR(datainicio) = 2018 THEN 1 ELSE 0 END)                       AS ano_2018,
    MAX(CASE WHEN YEAR(datainicio) = 2019 THEN 1 ELSE 0 END)                       AS ano_2019,
    MAX(CASE WHEN YEAR(datainicio) = 2020 THEN 1 ELSE 0 END)                       AS ano_2020,
    MAX(CASE WHEN YEAR(datainicio) = 2021 THEN 1 ELSE 0 END)                       AS ano_2021,
    MAX(CASE WHEN YEAR(datainicio) = 2022 THEN 1 ELSE 0 END)                       AS ano_2022,
    MAX(CASE WHEN YEAR(datainicio) = 2023 THEN 1 ELSE 0 END)                       AS ano_2023,
    MAX(CASE WHEN YEAR(datainicio) = 2024 THEN 1 ELSE 0 END)                       AS ano_2024,
    MAX(CASE WHEN YEAR(datainicio) = 2025 THEN 1 ELSE 0 END)                       AS ano_2025,
    MAX(CASE WHEN YEAR(datainicio) = 2026 THEN 1 ELSE 0 END)                       AS ano_2026
FROM stg_estatistico.siplan_data
GROUP BY atividade_id
'''

In [13]:
MONTH_ABBR = {1:'jan',2:'fev',3:'mar',4:'abr',5:'mai',6:'jun',
              7:'jul',8:'ago',9:'set',10:'out',11:'nov',12:'dez'}

def transform_datas(df: pd.DataFrame, rps_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- data/hora da primeira sessão ---
    df['primeiradata'] = pd.to_datetime(df['primeiradata'], errors='coerce')
    df['PrimeiraData']     = df['primeiradata'].dt.date
    df['PrimeiraHora']     = df['primeiradata'].dt.time
    df['PrimeiraDataHora'] = df['primeiradata'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['mes'] = df['primeiradata'].dt.month.map(MONTH_ABBR)

    # --- tempo médio da sessão (truncado a 1 decimal) ---
    df['tempo_da_sessao'] = (
        np.floor(pd.to_numeric(df['tempo_sessao'], errors='coerce').fillna(0) * 10) / 10
    )

    # --- flags de prazo (binário → 'sim'/'0') ---
    # 30dias: nº de datas distintas com sessão > 30  (≠ diascorridos > 30)
    # 90dias: diascorridos > 90
    # 60horas: total de horas > 60
    # Todos já calculados no SQL; apenas mapeamos para 'sim'/'0'
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    for col in ['30dias', '90dias', '60horas']:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(bool_map).fillna('0')

    df['ExtrapolaDataHora'] = np.where(
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'), 'sim', '0'
    )

    # --- autonomia temporal bruta ---
    # DIREG: ultrapassa 90 dias corridos OU 60 horas totais
    # STS  : mais de 30 datas distintas com sessão
    # UO   : demais casos
    conditions = [
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'),
        df['30dias'] == 'sim',
    ]
    df['autonomiaTemporal'] = np.select(conditions, ['DIREG', 'STS'], default='UO')

    # --- ajuste por serviço (via RPS Parcial) ---
    # A autonomia temporal só faz sentido para serviços com acúmulo de carga/dias.
    # Para os demais, UO é sempre o nível correto independente das datas.
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    usa_temporal = (
        df['servico'].isin({'Curso', 'Ioga'}) |
        df['servico'].str.startswith('Desenvolvimento') |
        df['subatividade'].str.startswith('Ações')
    )
    df['autonomiaTemporal'] = np.where(usa_temporal, df['autonomiaTemporal'], 'UO')

    # servico/subatividade já estarão na tabela base; remove daqui para evitar duplicatas
    df = df.drop(columns=['servico', 'subatividade'])

    return df


datas_df = transform_datas(query_to_df(sql_datas), rps_parcial_df)
datas_df.shape

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_20632\2510096508.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


(449880, 27)

## 3. Contratos (siplan_solicitacao + siplan_acao)

Solicitações administrativas de contratação por `atividade_id`, com custos, público e métricas calculadas.

Filtra `solicitacao_area = 'Administrativo'` com `datainicio > DATA_INICIAL` e `item_grupo`/`nome_item`
contendo: **contrato, passagem, hospedagem ou filme**.

**Campos vindos do SQL:**
- `solicitacao_id`, `grupo`, `item`, `complemento`, `custo` — identificação e custo da solicitação individual
- `publico`, `capacidade`, `estimativa`, `tipo_per_capita` — público planejado (de `siplan_acao`)
- `custo_contratos_total` — soma de custos **apenas de contrato+filme** da atividade (exclui passagem/hospedagem)
- `custo_total` — soma de **todas** as solicitações da atividade
- `n_contratos` — contagem de solicitações de contrato+filme
- `n_solic` — contagem total de solicitações

**Transformações aplicadas:**
- `sessoes`, `horas` — vindos de `datas_df` (dados reais de `siplan_data`)
- `acima15mil`, `acima20mil`, `acima100mil` — flags baseadas em `custo` da **solicitação individual**
- `publico_sessao` (renomeado de `publico`): público por sessão conforme cadastro
- `publico` (recalculado): público total
  - `tipo_per_capita = 0` → `sessoes × publico_sessao`
  - caso contrário → `publico_sessao` (já é total)
- `por_sessao`, `por_hora`, `per_capita` — calculados sobre `custo_contratos_total` (total da atividade)
- `grupo` — limpeza de caracteres extras no campo
- Merge com `rps_parcial_df` → `servico`, `subatividade`
- `autonomiaCusto` — baseada nos flags do `custo` individual: DIREG (>100k), STS/STS-20 (>20k), STS-15 (>15k), UO
- `por_hora_valido` — `por_hora` apenas para serviços/subatividades em `SERVICOS_POR_HORA`/`SUBATIV_POR_HORA`
- Deduplicação por `solicitacao_id`

> Lakehouse Staging: **`raw_contratos`**


In [17]:
# Lakehouse Staging: raw_contratos
sql_contratos = f'''
SELECT
    solic.atividade_id,
    solic.solicitacao_id,
    solic.item_grupo                                                    AS grupo,
    solic.nome_item                                                     AS item,
    solic.descricao                                                     AS complemento,
    solic.custo,
    acao.publico,
    acao.lugares                                                        AS capacidade,
    acao.estimativa_publico                                             AS estimativa,
    acao.tipo_per_capita,
    NVL(solic_contratos_total.custo_contratos_total, 0)                 AS custo_contratos_total,
    NVL(solic_total.custo_total, 0)                                     AS custo_total,
    NVL(contagem_contratos.n_contratos, 0)                              AS n_contratos,
    NVL(contagem_total.n_solic, 0)                                      AS n_solic

FROM (
    SELECT
        atividade_id,
        NVL(solicitacao_id, 0) AS solicitacao_id,
        NVL(custo, 0) AS custo,
        item_grupo,
        nome_item,
        descricao
    FROM stg_estatistico.siplan_solicitacao
    WHERE solicitacao_area = 'Administrativo'
      AND (   LOWER(item_grupo) LIKE '%contrato%'
           OR LOWER(nome_item) LIKE '%contrato%'
           OR LOWER(nome_item) LIKE '%passagem%'
           OR LOWER(nome_item) LIKE '%hospedagem%'
           OR LOWER(item_grupo) LIKE '%passagem%'
           OR LOWER(item_grupo) LIKE '%hospedagem%'
           OR LOWER(item_grupo) LIKE '%filme%')
      AND datainicio > '{DATA_INICIAL}'
) solic

INNER JOIN (
    SELECT DISTINCT
        atividade_id,
        lugares,
        estimativa_publico,
        CASE WHEN NVL(lugares, NVL(estimativa_publico, 1)) <= 0 THEN 1
             ELSE NVL(lugares, NVL(estimativa_publico, 1)) END          AS publico,
        CASE WHEN LOWER(desc_realizacao) IN ('curso', 'seminário') THEN 1
             ELSE 0 END                                                 AS tipo_per_capita
    FROM stg_estatistico.siplan_acao
) acao ON solic.atividade_id = acao.atividade_id

LEFT JOIN (
    SELECT atividade_id, SUM(NVL(custo, 0)) AS custo_contratos_total
    FROM stg_estatistico.siplan_solicitacao
    WHERE solicitacao_area = 'Administrativo'
      AND (   LOWER(item_grupo) LIKE '%contrato%'
           OR LOWER(nome_item)  LIKE '%contrato%'
           OR LOWER(item_grupo) LIKE '%filme%')
      -- par semantico de n_contratos: passagens/hospedagens contribuem via custo individual
      AND datainicio > '{DATA_INICIAL}'
    GROUP BY atividade_id
) solic_contratos_total ON solic.atividade_id = solic_contratos_total.atividade_id

LEFT JOIN (
    SELECT atividade_id, SUM(NVL(custo, 0)) AS custo_total
    FROM stg_estatistico.siplan_solicitacao
    WHERE datainicio > '{DATA_INICIAL}'
    GROUP BY atividade_id
) solic_total ON solic.atividade_id = solic_total.atividade_id

LEFT JOIN (
    SELECT atividade_id, COUNT(*) AS n_contratos
    FROM stg_estatistico.siplan_solicitacao
    WHERE solicitacao_area = 'Administrativo'
      AND (   LOWER(item_grupo) LIKE '%contrato%'
           OR LOWER(nome_item)  LIKE '%contrato%'
           OR LOWER(item_grupo) LIKE '%filme%')
      AND datainicio > '{DATA_INICIAL}'
    GROUP BY atividade_id
) contagem_contratos ON solic.atividade_id = contagem_contratos.atividade_id

LEFT JOIN (
    SELECT atividade_id, COUNT(*) AS n_solic
    FROM stg_estatistico.siplan_solicitacao
    WHERE datainicio > '{DATA_INICIAL}'
    GROUP BY atividade_id
) contagem_total ON solic.atividade_id = contagem_total.atividade_id
'''

In [18]:
SERVICOS_LIMIAR_20K  = {'Debate', 'Seminário', 'Visita Mediada'}
SUBATIV_LIMIAR_20K   = {'Ações formativas', 'Ações mediadas', 'Passeios', 'Viagens'}
SERVICOS_POR_HORA    = {'Curso', 'Oficina', 'Vivência', 'Seminário', 'Mediação', 'Visita Mediada', 'Intervenção urbana'}
SUBATIV_POR_HORA     = {'Multipráticas recreativas', 'Passeios', 'Viagens', 'Colônias recreativas'}

def transform_contracts(df: pd.DataFrame, rps_df: pd.DataFrame, datas_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- sessoes e horas reais (de datas_df, que vem de siplan_data) ---
    df = df.merge(
        datas_df[['atividade_id', 'qt_sessoes', 'qt_horas']],
        on='atividade_id', how='left'
    )
    df = df.rename(columns={'qt_sessoes': 'sessoes', 'qt_horas': 'horas'})
    df['sessoes'] = pd.to_numeric(df['sessoes'], errors='coerce').fillna(0)
    df['horas']   = pd.to_numeric(df['horas'],   errors='coerce').fillna(0)

    # --- flags de custo (baseados no custo da solicitação individual, binário → 'sim'/'0') ---
    custo_individual = pd.to_numeric(df['custo'], errors='coerce').fillna(0)
    custo_total      = pd.to_numeric(df['custo_contratos_total'], errors='coerce').fillna(0)
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    df['acima15mil']  = (custo_individual > 15000).astype(int).map(bool_map)
    df['acima20mil']  = (custo_individual > 20000).astype(int).map(bool_map)
    df['acima100mil'] = (custo_individual > 100000).astype(int).map(bool_map)

    # --- público total ---
    # tipo_per_capita = 0 → público se repete por sessão → total = sessoes × publico_sessao
    # tipo_per_capita = 1 → público já é total
    df = df.rename(columns={'publico': 'publico_sessao'})
    pub = pd.to_numeric(df['publico_sessao'],  errors='coerce').fillna(0)
    tpc = pd.to_numeric(df['tipo_per_capita'], errors='coerce').fillna(0)
    df['publico'] = np.where(pub > 4000, pub, np.where(tpc == 0, df['sessoes'] * pub, pub)).astype(int)

    # --- custo por sessão, por hora e per capita (sobre custo_contratos_total da atividade) ---
    df['por_sessao'] = np.where(df['sessoes'] > 0, custo_total / df['sessoes'], np.nan)
    df['por_hora']   = np.where(df['horas']   > 0, custo_total / df['horas'],   np.nan)
    df['per_capita'] = np.where(
        tpc == 0,
        np.where(df['publico'] > 0, custo_total / df['publico'], np.nan),
        np.where(pub > 0, custo_total / pub, np.nan)
    )

    # --- limpeza do grupo (remove sufixo "[...]" inserido pela view) ---
    df['grupo'] = df['grupo'].fillna('').astype(str).str.split('[').str[0].str.strip()

    # --- merge com RPS Parcial → servico e subatividade ---
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # --- autonomia de custo (baseada no custo individual da solicitação) ---
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'

    conditions = [
        acima100,
        acima20 &  limiar_20k,
        acima20 & ~limiar_20k,
        acima15 & ~limiar_20k,
    ]
    df['autonomiaCusto'] = np.select(conditions, ['DIREG', 'STS', 'STS-20', 'STS-15'], default='UO')

    # --- por_hora válido apenas para serviços que usam custo por hora ---
    usa_por_hora = (
        df['servico'].isin(SERVICOS_POR_HORA) |
        df['subatividade'].isin(SUBATIV_POR_HORA)
    )
    df['por_hora_valido'] = np.where(usa_por_hora, df['por_hora'], np.nan)

    # --- uma linha por solicitação ---
    df = df.drop_duplicates(subset=['solicitacao_id'])

    return df


contracts_df = transform_contracts(query_to_df(sql_contratos), rps_parcial_df, datas_df)
contracts_df.shape

C:\Users\sergio.seabra\AppData\Local\Temp\ipykernel_20632\2510096508.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


(20527, 27)

## 4. Autonomias

Combina as três tabelas anteriores para calcular a `autonomia` final de cada atividade.

**Fontes e o que contribuem:**
| Fonte | Campo | Regra |
|---|---|---|
| Datas | `autonomiaTemporal` | DIREG (90d/60h), STS (30d corridos), UO — ajustado por serviço |
| Contratos | `autonomiaCusto` | DIREG (>100k), STS-20/STS (20k–100k), STS-15 (15k–20k), UO |
| RPS Parcial | `tag` | STS se `'Avaliação STS'` |
| RPS Parcial | `tem_passagem` | STS se `'Sim'` |

**Hierarquia:** `DIREG > STS > UO` — prevalece sempre o maior nível entre todas as fontes.

`autonomiaCusto` preserva o qualificador (`STS-15`, `STS-20`) para rastreabilidade, mas é tratado como nível STS na hierarquia.

In [19]:
# Hierarquia: DIREG (3) > STS (2) > UO (1)
# Cada fonte de autonomia é convertida para nível numérico;
# o nível máximo determina a autonomia final.

NIVEL = {'DIREG': 3, 'STS': 2, 'STS-20': 2, 'STS-15': 2, 'UO': 1}

def autonomia_nivel(serie: pd.Series) -> pd.Series:
    """Converte valores de autonomia para nível numérico (desconhecido → 1)."""
    return serie.map(NIVEL).fillna(1).astype(int)

NIVEL_INV = {3: 'DIREG', 2: 'STS', 1: 'UO'}

def build_autonomias(rps_df, datas_df, contracts_df) -> pd.DataFrame:
    # Base: uma linha por atividade com serviço cadastrado
    df = rps_df[['atividade_id', 'servico', 'subatividade', 'tag', 'tem_passagem']].copy()

    # --- autonomia temporal (já ajustada por serviço em Datas) ---
    df = df.merge(
        datas_df[['atividade_id', 'autonomiaTemporal']],
        on='atividade_id', how='left'
    )
    df['autonomiaTemporal'] = df['autonomiaTemporal'].fillna('UO')

    # --- autonomia de custo (uma linha por atividade) ---
    custo_por_ativ = (
        contracts_df[['atividade_id', 'autonomiaCusto']]
        .drop_duplicates(subset=['atividade_id'])
    )
    df = df.merge(custo_por_ativ, on='atividade_id', how='left')
    df['autonomiaCusto'] = df['autonomiaCusto'].fillna('UO')

    # --- níveis por fonte ---
    nivel_temporal  = autonomia_nivel(df['autonomiaTemporal'])
    nivel_custo     = autonomia_nivel(df['autonomiaCusto'])
    nivel_tag       = np.where(df['tag'] == 'Avaliação STS', 2, 1)
    nivel_passagem  = np.where(df['tem_passagem'] == 'Sim',  2, 1)

    # --- autonomia final: prevalece o maior nível (DIREG > STS > UO) ---
    nivel_final = pd.concat(
        [nivel_temporal, nivel_custo,
         pd.Series(nivel_tag, index=df.index),
         pd.Series(nivel_passagem, index=df.index)],
        axis=1
    ).max(axis=1)

    df['autonomia'] = nivel_final.map(NIVEL_INV)

    return df


autonomias_df = build_autonomias(rps_parcial_df, datas_df, contracts_df)
autonomias_df['autonomia'].value_counts()

autonomia
UO       27780
STS       3765
DIREG     1531
Name: count, dtype: int64

## 5. Projetos (siplan_acao + projetos + projeto_uo)

Metadados de projeto por `atividade_id`. Cada linha = uma atividade (deduplicada).

**Campos produzidos:**
- `projeto_id`, `projeto_nome`, `projeto_complemento` — identificação do projeto
- `institucional`, `projeto_categoria`, `tag_projeto`, `tag_grupo_projeto` — classificação
- `projeto_uo_nome`, `projeto_descricao`, `projeto_comunicacao`, `projeto_conceitual` — dados da UO
- `tem_pai` (bool) — indica se o projeto tem pai hierárquico em `projeto_uo`

**Decisões de design:**
- Join via `projeto_uo.id` cobre 100% dos projetos; 409 atividades com `projeto_id` órfão preservadas via LEFT JOIN
- JOIN com tabela pai removido — `tem_pai` é suficiente para derivar flags como `espaco_brincar`
- Filtro pós-load para manter apenas atividades do período corrente (`raw_acoes_df`)

> Lakehouse Staging: **`raw_projetos`**


In [ ]:
# Lakehouse Staging: raw_projetos
sql_raw_projetos = f'''
SELECT
    a.atividade_id,

    -- Metadados do projeto (de projetos)
    p.projeto_id,
    p.projeto                       AS projeto_nome,
    p.projeto_complemento           AS projeto_complemento,
    p.institucional                 AS institucional,
    p.categoria_projeto             AS projeto_categoria,
    p.tag_projeto                   AS tag_projeto,
    p.tag_grupo_projeto             AS tag_grupo_projeto,

    -- Projeto UO — nível próprio
    pu.nome                         AS projeto_uo_nome,
    pu.descricao                    AS projeto_descricao,
    pu.descricao_comunicacao        AS projeto_comunicacao,
    pu.descricao_conceitual         AS projeto_conceitual,
    CASE WHEN pu.projetopai_id IS NOT NULL THEN TRUE ELSE FALSE END AS tem_pai

FROM stg_estatistico.siplan_acao a
LEFT JOIN stg_estatistico.projetos p
       ON a.projeto_id = p.projeto_id
LEFT JOIN stg_estatistico.projeto_uo pu
       ON p.projeto_id = pu.id
WHERE a.projeto_id IS NOT NULL
  AND a.atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''

In [ ]:
raw_projetos_df = query_to_df(sql_raw_projetos)
raw_projetos_df.columns = [col.split(".")[-1] for col in raw_projetos_df.columns]
raw_projetos_df["atividade_id"] = raw_projetos_df["atividade_id"].astype(str).str.strip()

# Filtra apenas atividades do período corrente (já carregadas em raw_acoes_df)
raw_projetos_df = raw_projetos_df[
    raw_projetos_df["atividade_id"].isin(raw_acoes_df["atividade_id"])
]

# siplan_acao pode ter múltiplas linhas por atividade_id;
# os dados de projeto são iguais em todas — mantém só a primeira
raw_projetos_df = raw_projetos_df.drop_duplicates(subset=["atividade_id"])

# Normaliza coluna de nome
for col in ["projeto_uo_nome"]:
    if col in raw_projetos_df.columns:
        raw_projetos_df[col] = raw_projetos_df[col].fillna("").astype(str).str.strip()

print(f"raw_projetos_df: {raw_projetos_df.shape}")
print(f"atividades com projeto: {raw_projetos_df['atividade_id'].nunique()}")
print(f"com pai: {raw_projetos_df['tem_pai'].sum()}")

## 6. Datas / Sessões (siplan_data)

Sessões de atividades com local, horário, tipologia e derivações temporais.
Substitui a query `raw_datas_sessoes` do Power Query. Cada linha = uma sessão.

O JOIN com subquery traz `primeira_data` e `primeira_hora` (data e hora da primeira sessão de cada atividade).

**Campos produzidos:**
- `atividade_id`, `sessao_id`, `uo` — identificação
- `datainicio`, `Data` (datetime meia-noite), `diferenca` (horas inteiras), `Duracao` (timedelta) — tempo
- `HoraCheia` (hora cheia), `horaCerta` (hora exata), `horaCertaTxt` (texto HH:MM:SS) — horário
- `PeriodoDia` — Madrugada (0–4h) / de manhã (5–11h) / à tarde (12–17h) / à noite (18–23h)
- `local`, `TipologiaLocal` (ex-`grupo_nome`), `TipoLocal` (`online`/`externa`/`na UO`) — local
- `ano`, `Mes`, `mesTxt`, `dia`, `diaSemama`, `diaSemanaTxt`, `Semana do Ano` — partes de data
- `primeira_data`, `primeira_hora`, `mes1o`, `mes1oTxt` — mês de referência da atividade

**Otimizações vs. Power Query:**
- Filtro de data feito apenas no SQL (Power Query filtrava duas vezes)
- `TipoLocal` calculado com máscaras vetorizadas em vez de `if/elif` por linha
- `PeriodoDia` via `np.select` (equivalente ao `SWITCH(TRUE(),...)` do DAX)

> Lakehouse Staging: **`raw_datas_sessoes`**


In [ ]:
# Lakehouse Staging: raw_datas_sessoes
sql_raw_datas_sessoes = f'''
SELECT
    d.sessao_id,
    d.atividade_id,
    d.datainicio,
    d.datafinal,
    d.local_id,
    d.local_nome,
    d.grupo_id,
    d.grupo_nome,
    d.geac,
    d.correcao_local,
    d.uo_local,
    TO_DATE(m.primeira_ts)                  AS primeira_data,
    date_format(m.primeira_ts, 'HH:mm:ss')  AS primeira_hora
FROM stg_estatistico.siplan_data AS d
JOIN (
    SELECT
        atividade_id,
        MIN(datainicio) AS primeira_ts
    FROM stg_estatistico.siplan_data
    GROUP BY atividade_id
) AS m ON m.atividade_id = d.atividade_id
WHERE TO_DATE(d.datainicio) >= DATE '{DATA_INICIAL}'
'''


In [ ]:
# Nota: diaSemama preserva o typo original do Power Query (era "semana").
# Renomear quebraria relatórios existentes que referenciem essa coluna.
WEEKDAY_ABBR = {0: "seg", 1: "ter", 2: "qua", 3: "qui",
                4: "sex", 5: "sab", 6: "dom"}


def transform_datas_sessoes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [col.split(".")[-1] for col in df.columns]

    # Tipos base
    df["atividade_id"]  = df["atividade_id"].astype(str).str.strip()
    df["sessao_id"]     = df["sessao_id"].astype(str).str.strip()
    df["datainicio"]    = pd.to_datetime(df["datainicio"],    errors="coerce")
    df["datafinal"]     = pd.to_datetime(df["datafinal"],     errors="coerce")
    df["primeira_data"] = pd.to_datetime(df["primeira_data"], errors="coerce")

    # Remove sessões com datas fora do intervalo datetime64[ns] (ex: anos > 2262)
    n_before = len(df)
    df = df[df["datainicio"].notna()].copy()
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"[AVISO] {n_dropped} sessão(ões) descartada(s): datainicio inválida ou fora dos limites")

    # Exclui grupo "37" (tipologia sem validade)
    df = df[df["grupo_nome"] != "37"].copy()

    # Duração e diferença em horas inteiras
    df["Duracao"]   = df["datafinal"] - df["datainicio"]
    df["diferenca"] = (df["Duracao"].dt.total_seconds() / 3600).astype(int)

    # Colunas de tempo derivadas de datainicio
    dt = df["datainicio"].dt
    df["Data"]         = dt.normalize()                        # datetime na meia-noite
    df["HoraCheia"]    = df["datainicio"].dt.floor("h").dt.time  # hora cheia (sem minutos)
    df["horaCerta"]    = dt.time                               # hora exata
    df["horaCertaTxt"] = dt.strftime("%H:%M:%S")

    # PeriodoDia baseado na hora cheia
    hora = dt.hour
    df["PeriodoDia"] = np.select(
        [hora < 5, hora < 12, hora < 18],
        ["Madrugada", "de manhã", "à tarde"],
        default="à noite"
    )

    # TipoLocal — vetorizado (máscara de maior prioridade aplicada por último)
    g = df["grupo_nome"].fillna("")
    l = df["local_nome"].fillna("")
    df["TipoLocal"] = "na UO"
    df.loc[g == "Fora da Unidade",              "TipoLocal"] = "externa"
    df.loc[l.str.contains("Online",       na=False), "TipoLocal"] = "online"
    df.loc[l.str.contains("Sesc Digital", na=False), "TipoLocal"] = "online"
    df.loc[g.str.contains("Online",       na=False), "TipoLocal"] = "online"

    # Renomeia e formata texto
    df = df.rename(columns={"grupo_nome": "TipologiaLocal", "local_nome": "local"})
    df["local"] = df["local"].str.title()

    # uo = primeiros 2 dígitos do atividade_id
    df["uo"] = df["atividade_id"].str[:2].astype(int)

    # Partes de data (de datainicio)
    df["ano"]           = dt.year
    df["Mes"]           = dt.month
    df["mesTxt"]        = dt.month.map(MONTH_ABBR)
    df["dia"]           = dt.day
    df["diaSemama"]     = dt.dayofweek             # seg=0, dom=6
    df["diaSemanaTxt"]  = dt.dayofweek.map(WEEKDAY_ABBR)
    df["Semana do Ano"] = dt.isocalendar().week.astype(int)

    # Partes de data (de primeira_data — mês da 1ª sessão da atividade)
    p = df["primeira_data"].dt
    df["mes1o"]    = p.month
    df["mes1oTxt"] = p.month.map(MONTH_ABBR)

    # Remove colunas não consumidas downstream
    df = df.drop(columns=["local_id", "grupo_id", "correcao_local",
                          "uo_local", "geac", "datafinal"], errors="ignore")

    col_order = [
        "atividade_id", "sessao_id", "uo",
        "datainicio", "Data", "diferenca", "Duracao",
        "HoraCheia", "horaCerta", "horaCertaTxt", "PeriodoDia",
        "local", "TipologiaLocal", "TipoLocal",
        "ano", "Mes", "mesTxt", "dia", "diaSemama", "diaSemanaTxt", "Semana do Ano",
        "primeira_data", "primeira_hora", "mes1o", "mes1oTxt",
    ]
    return df[[c for c in col_order if c in df.columns]]


In [ ]:
raw_datas_sessoes_df = query_to_df(sql_raw_datas_sessoes)
raw_datas_sessoes_df = transform_datas_sessoes(raw_datas_sessoes_df)

print(f'raw_datas_sessoes_df: {raw_datas_sessoes_df.shape}')
print(f'atividades únicas:     {raw_datas_sessoes_df["atividade_id"].nunique()}')
print(f'\nTipoLocal:')
print(raw_datas_sessoes_df['TipoLocal'].value_counts())


## 7. Acessibilidade (siplan_acessibilidade)

Recursos de acessibilidade vinculados a cada `atividade_id`.
Usada em dois contextos:
- **Relatórios** — detalha quais recursos estão associados a cada atividade
- **Tabela base** — campo derivado `tem_dispositivo` (sim/não): qualquer presença nesta tabela implica "sim"

**Campos:**
- `atividade_id`, `acessibilidade_id`, `identificacao` — direto do banco
- `tem_dispositivo` — sempre "sim" (o filtro `acessibilidade_id <> 9` já exclui os registros sem dispositivo)
- `uo` — primeiros 2 dígitos do `atividade_id`


In [ ]:
# Lakehouse Staging: raw_acessibilidade
sql_raw_acessibilidade = f'''
SELECT
    atividade_id,
    acessibilidade_id,
    identificacao,
    CASE
        WHEN acessibilidade_id = 9 THEN 'não' ELSE 'sim'
    END AS tem_dispositivo
FROM stg_estatistico.siplan_acessibilidade
WHERE acessibilidade_id <> 9
  AND acessibilidade_id IS NOT NULL
  AND atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''


In [ ]:
raw_acessibilidade_df = query_to_df(sql_raw_acessibilidade)
raw_acessibilidade_df.columns = [col.split('.')[-1] for col in raw_acessibilidade_df.columns]
raw_acessibilidade_df['atividade_id'] = raw_acessibilidade_df['atividade_id'].astype(str).str.strip()
raw_acessibilidade_df['uo']           = raw_acessibilidade_df['atividade_id'].str[:2].astype(int)

# Filtra apenas atividades do período corrente
raw_acessibilidade_df = raw_acessibilidade_df[
    raw_acessibilidade_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()

print(f'raw_acessibilidade_df: {raw_acessibilidade_df.shape}')
print(f'atividades únicas:     {raw_acessibilidade_df["atividade_id"].nunique()}')
print()
print('recursos:')
print(raw_acessibilidade_df['identificacao'].value_counts())


## 8. Solicitações (siplan_solicitacao)

Staging pura de todas as solicitações com `custo > 0` para o período corrente.
Cada linha = uma solicitação; várias por `atividade_id`.

**Campos:** `atividade_id`, `solicitacao_id`, `custo`, `area`, `item_grupo`, `nome_item`, `descricao`, `status_solic`, `obs_solicitacao`

> Fonte para as saídas transformadas (8a e futuras). Substituirá `raw_contratos` (seção 3) após revisão.


In [ ]:
# Lakehouse Staging: raw_solicitacoes
sql_raw_solicitacoes = f'''
SELECT
    solic.atividade_id,
    NVL(solic.solicitacao_id, 0)        AS solicitacao_id,
    NVL(solic.custo, 0)                 AS custo,
    solic.solicitacao_area              AS area,
    solic.item_grupo                    AS item_grupo,
    solic.nome_item                     AS nome_item,
    solic.descricao                     AS descricao,
    solic.status_solicitacao            AS status_solic,
    solic.obs_solicitacao               AS obs_solicitacao
FROM stg_estatistico.siplan_solicitacao solic
WHERE solic.datainicio > '{DATA_INICIAL}'
  AND NVL(solic.custo, 0) > 0
  AND solic.atividade_id IN (
      SELECT DISTINCT atividade_id
      FROM stg_estatistico.siplan_data
      WHERE datainicio > '{DATA_INICIAL}'
  )
'''


In [ ]:
raw_solicitacoes_df = query_to_df(sql_raw_solicitacoes)
raw_solicitacoes_df.columns = [col.split('.')[-1] for col in raw_solicitacoes_df.columns]
raw_solicitacoes_df['atividade_id']   = raw_solicitacoes_df['atividade_id'].astype(str).str.strip()
raw_solicitacoes_df['solicitacao_id'] = raw_solicitacoes_df['solicitacao_id'].astype(str).str.strip()
raw_solicitacoes_df['custo']          = pd.to_numeric(raw_solicitacoes_df['custo'], errors='coerce').fillna(0)

# Filtra apenas atividades do período corrente
raw_solicitacoes_df = raw_solicitacoes_df[
    raw_solicitacoes_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()

print(f'raw_solicitacoes_df: {raw_solicitacoes_df.shape}')
print(f'atividades unicas:   {raw_solicitacoes_df["atividade_id"].nunique()}')
print()
print('areas:')
print(raw_solicitacoes_df['area'].value_counts())


## 8a. Descrição de custos por atividade (solicitacoes_desc_df)

Uma linha por `atividade_id`; campo `item_desc` com todas as solicitações concatenadas em texto.

**`item_de_custo` → `item_norm`** (normalização):
- Se `item_grupo` for nulo ou `'null'`: usa `nome_item` até o primeiro `'-'` (com trim)
- Mapeamento via `MAPA_ITEM_CUSTO` (match exato) + `PREFIXOS_ITEM_CUSTO` (fallback por prefixo)
- `item_grupo = 'acessibilidade'` → `Acessibilidade` (independente do nome)

**Descartados:** `Camarim` (tipos 1/2), `Água`, `Verificar`, `Estagiário`

**Categorias finais:** `Contrato PJ` · `Contrato PF` · `Contrato Cooperativa` · `Hospedagem` · `Passagem Aérea` · `Turismo` · `Transporte` · `Exibição de Filmes` · `Ação esportiva e recreativa` · `Contratações diversas` · `Compras` · `Alimentação` · `Sonorização` · `Iluminação` · `Audiovisual` · `Locação` · `Comunicação` · `Acessibilidade` · `Outros`

**Ordem em `item_desc`:**
1. Contratos (PJ / PF / Cooperativa)
2. `────────────────────────────`
3. Passagem Aérea
4. `────────────────────────────`
5. Hospedagem
6. `────────────────────────────`
7. Demais categorias — ordem alfabética por `item_norm`, depois `solicitacao_id`

**Saída:** `atividade_id`, `item_desc`


In [ ]:
# Itens a descartar antes de construir item_desc
ITENS_DESCARTAR = frozenset([
    'Camarim', 'Camarim Tipo 1', 'Camarim Tipo 2',
    'Água', 'Verificar',
])

# Mapeamento exato: item_de_custo → categoria normalizada
MAPA_ITEM_CUSTO = {
    # Contratos
    'Contrato PJ [Custo cachê/pró-labore]':            'Contrato PJ',
    'Contrato PJ':                                      'Contrato PJ',
    'Contrato PF [Custo cachê/pró-labore]':            'Contrato PF',
    'Contrato PF':                                      'Contrato PF',
    'Contrato Cooperativa [Custo cachê/pró-labore]':   'Contrato Cooperativa',
    # Viagem
    'Hospedagem [Custo hospedagem]':                   'Hospedagem',
    'Passagem Aérea [Custo passagem]':                 'Passagem Aérea',
    'Turismo [Outros custos de terceiros]':            'Turismo',
    'Turismo':                                          'Turismo',
    'Transporte [Outros custos de terceiros]':         'Transporte',
    'Transporte':                                       'Transporte',
    # Serviços artísticos/técnicos
    'Exibição de Filmes':                              'Exibição de Filmes',
    'Ação esportiva e recreativa':                     'Ação esportiva e recreativa',
    'Contratações diversas [Outros custos de terceiros]': 'Contratações diversas',
    # Compras
    'Compras  [Outros custos internos]':               'Compras',
    'Compras':                                          'Compras',
    'Aquisição de material':                           'Compras',
    # Alimentação
    'Kit Lanche':                                       'Alimentação',
    'Brindes':                                          'Alimentação',
    'Café para integrações/Ações Similares':           'Alimentação',
    'Coffee break Tipo 1':                             'Alimentação',
    'Coffee break Tipo 2':                             'Alimentação',
    'Coquetel':                                         'Alimentação',
    'Recepções / Refeições':                           'Alimentação',
    'Diversos - Alimentação [Outros custos de terceiros]': 'Alimentação',
    # Infraestrutura de evento
    'Sonorização':                                      'Sonorização',
    'Locação - Sonorização [Outros custos de terceiros]': 'Sonorização',
    'Iluminação':                                       'Iluminação',
    'Locação - Iluminação [Outros custos de terceiros]': 'Iluminação',
    'Audiovisual / Projeção':                          'Audiovisual',
    'Audiovisual':                                      'Audiovisual',
    'Locação - Outros [Outros custos de terceiros]':   'Locação',
    'Locação':                                          'Locação',
    # Comunicação
    'Impressos e digitais':                            'Comunicação',
    'Editoria web':                                     'Comunicação',
    'Assessoria de imprensa':                          'Comunicação',
    'Diversos - Comunicação [Outros custos de terceiros]': 'Comunicação',
    # Acessibilidade
    'Tradução Simultânea':                             'Acessibilidade',
    # Outros (categorias absorvidas)
    'Serviço de Receptivo':                            'Outros',
    'Mobiliário':                                       'Outros',
    'Montagem':                                         'Outros',
    'Equipamentos':                                     'Outros',
    'Limpeza':                                          'Outros',
    'Elétrica':                                         'Outros',
    'Acompanhamento':                                   'Outros',
    'Outros':                                           'Outros',
    'Outros - Terceiros [Outros custos terceiros]':    'Outros',
    'Outros - Internos [Outros custos internos]':      'Outros',
}

# Fallback por prefixo — para itens com código de conta no nome (ex: 'Hospedagem [3920')
PREFIXOS_ITEM_CUSTO = [
    ('Hospedagem',           'Hospedagem'),
    ('Guia de turismo',      'Turismo'),
    ('Locação de Automóvel', 'Transporte'),
    ('Serviço de Montagem',  'Outros'),
    ('Serviços Gerais',      'Outros'),
]


def normalizar_item_custo(idc: str, item_grupo: str) -> str:
    """Retorna categoria normalizada; None = descartar."""
    idc_s   = str(idc).strip()
    grupo_l = str(item_grupo).strip().lower()
    # Estagiário → descarte
    if 'estagi' in grupo_l or 'estagi' in idc_s.lower():
        return None
    # item_grupo = 'acessibilidade' → Acessibilidade
    if grupo_l == 'acessibilidade':
        return 'Acessibilidade'
    # Match exato
    if idc_s in MAPA_ITEM_CUSTO:
        return MAPA_ITEM_CUSTO[idc_s]
    # Fallback por prefixo
    for prefixo, categoria in PREFIXOS_ITEM_CUSTO:
        if idc_s.startswith(prefixo):
            return categoria
    # Sem mapeamento — mantém o valor bruto
    return idc_s


In [ ]:
# Ordem de exibição dos grupos em item_desc
# 0 = Contratos  1 = Passagem Aérea  2 = Hospedagem  3 = demais (alfabético)
GRUPO_ITEM_DESC = {
    'Contrato PJ':          0,
    'Contrato PF':          0,
    'Contrato Cooperativa': 0,
    'Passagem Aérea':       1,
    'Hospedagem':           2,
}
SEP_ITEM_DESC = chr(9472) * 28   # ────────────────────────────


def build_solicitacoes_desc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── item_de_custo bruto ───────────────────────────────────────────────
    grupo = df['item_grupo'].fillna('').astype(str).str.strip()
    nome  = df['nome_item'].fillna('').astype(str).str.strip()
    nome_ate_traco = nome.str.split('-').str[0].str.strip()
    usar_nome = (grupo == '') | (grupo.str.lower() == 'null')
    df['item_de_custo'] = np.where(usar_nome, nome_ate_traco, grupo)

    # ── descarte ─────────────────────────────────────────────────────────
    df = df[~df['item_de_custo'].isin(ITENS_DESCARTAR)].copy()

    # ── normalização ─────────────────────────────────────────────────────
    df['item_norm'] = df.apply(
        lambda r: normalizar_item_custo(r['item_de_custo'], r['item_grupo']),
        axis=1,
    )
    df = df[df['item_norm'].notna()].copy()

    # ── chave de ordenação: (grupo, item_norm alfabético, solicitacao_id) ─
    df['_grp'] = df['item_norm'].map(lambda x: GRUPO_ITEM_DESC.get(x, 3))

    # ── custo formatado ───────────────────────────────────────────────────
    df['custo_fmt'] = df['custo'].apply(
        lambda x: 'R$ ' + f'{x:,.0f}'.replace(',', '.')
    )

    # ── linha individual ──────────────────────────────────────────────────
    desc = df['descricao'].fillna('').astype(str).str.strip()
    df['linha'] = df['item_norm'] + ' — ' + desc + ' — ' + df['custo_fmt']

    # ── agrega por atividade_id com separadores entre grupos ─────────────
    def montar_desc(sub):
        sub = sub.sort_values(['_grp', 'item_norm', 'solicitacao_id'])
        linhas = []
        grp_atual = None
        for _, row in sub.iterrows():
            if grp_atual is not None and row['_grp'] != grp_atual:
                linhas.append(SEP_ITEM_DESC)
            linhas.append(row['linha'])
            grp_atual = row['_grp']
        return chr(10).join(linhas)

    resultado = (
        df.groupby('atividade_id')
        .apply(montar_desc)
        .reset_index(name='item_desc')
    )
    return resultado


solicitacoes_desc_df = build_solicitacoes_desc(raw_solicitacoes_df)

print(f'solicitacoes_desc_df: {solicitacoes_desc_df.shape}')
print()
print('Exemplo (primeira linha de item_desc):')
print(solicitacoes_desc_df['item_desc'].iloc[0])
